In [1]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.1 MB/s eta 0:00:00


# **Continuous**

# RUSA Tertiary

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/norm rusa ter rule based.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Shape of label tensor: (10016, 3)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding (Embedding)       (None, 100, 128)          3906816   
                                                                 
 transformer_block (Transfo  (None, 100, 128)          561024    
 rmerBlock)                                                      
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)           

#UCI Tertiary

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/norm uci ter rule based.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (20229, 3)
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_1 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_1 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_1  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_5 (Dense)         

## RUSA Bi

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/normalized rusa bi rule based.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (6726, 2)
Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_2 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_2 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_2  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_8 (Dropout)         (None, 128)               0         
                                                                 
 dense_8 (Dense)          

# UCI BI

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/norm uci bi rule based.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (11300, 2)
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_1 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_1 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_1  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_5 (Dense)         

# **RAW**

# RUSA Tertiary

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/RUSA-terraw.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (10016, 3)
Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_2 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_2 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_2  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_8 (Dropout)         (None, 128)               0         
                                                                 
 dense_8 (Dense)         

# UCI Tertiary

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/UCI-terraww.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Shape of label tensor: (20229, 3)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding (Embedding)       (None, 100, 128)          3906816   
                                                                 
 transformer_block (Transfo  (None, 100, 128)          561024    
 rmerBlock)                                                      
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)           

# RUSA BI

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/RUSA-biraw.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (6726, 2)
Model: "model_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_5 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_5 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_5  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_17 (Dropout)        (None, 128)               0         
                                                                 
 dense_17 (Dense)         

# UCI BI

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

df = pd.read_csv('/content/UCI-biraw.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (11300, 2)
Model: "model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_6 (Embedding)     (None, 100, 128)          3906816   
                                                                 
 transformer_block_6 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_6  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_20 (Dropout)        (None, 128)               0         
                                                                 
 dense_20 (Dense)        

# **Continuous**

# **Rusa Bi**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

df = pd.read_csv('/content/normalized rusa-bi.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim, input_length=MAX_SEQUENCE_LENGTH)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

Shape of label tensor: (6726, 2)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding (Embedding)       (None, 100, 128)          13552512  
                                                                 
 transformer_block (Transfo  (None, 100, 128)          561024    
 rmerBlock)                                                      
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)            

# **Rusa Ter**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

df = pd.read_csv('/content/normalized rusa-ter.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim, input_length=MAX_SEQUENCE_LENGTH)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (10016, 3)
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_1 (Embedding)     (None, 100, 128)          13552512  
                                                                 
 transformer_block_1 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_1  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_5 (Dense)         

# **UCI Bi**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

df = pd.read_csv('/content/normalized uci-bi.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim, input_length=MAX_SEQUENCE_LENGTH)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

Shape of label tensor: (11300, 2)
Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding_2 (Embedding)     (None, 100, 128)          13552512  
                                                                 
 transformer_block_2 (Trans  (None, 100, 128)          561024    
 formerBlock)                                                    
                                                                 
 global_average_pooling1d_2  (None, 128)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_8 (Dropout)         (None, 128)               0         
                                                                 
 dense_8 (Dense)         

# **UCI Ter**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

df = pd.read_csv('/content/normalized uci-ter.csv', encoding='cp437')
df['Comment']=df['Comment'].astype(str)
num_classes = df['sentiment'].nunique()

# The maximum number of words to be used. (most frequent)
MAX_SEQUENCE_LENGTH = 100

X=df['Comment']

y = pd.get_dummies(df['sentiment']).values
print('Shape of label tensor:', y.shape)

# Split data into training and test sets
seed = 35
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# Tokenization and sequence padding
X_train_tokens = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')
X_test_tokens = tokenizer(X_test.tolist(), padding='max_length', truncation=True, max_length=MAX_SEQUENCE_LENGTH, return_tensors='tf')

# Custom BERT model components
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(CustomMultiHeadAttention, self).__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

    def call(self, inputs):
        attn_output = self.mha(inputs, inputs)
        return attn_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = CustomMultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes):
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32)
    embedding_layer = Embedding(len(tokenizer), embed_dim, input_length=MAX_SEQUENCE_LENGTH)(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
    x = transformer_block(embedding_layer)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

embed_dim = 128
num_heads = 8
ff_dim = 128

model = build_custom_bert_model(embed_dim, num_heads, ff_dim, num_classes)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_train_tokens['input_ids'], y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
y_pred = model.predict(X_test_tokens['input_ids'])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print metrics
acc = accuracy_score(y_test_classes, y_pred_classes)
print(f'Accuracy: {acc}')
print(confusion_matrix(y_test_classes, y_pred_classes))
print(classification_report(y_test_classes, y_pred_classes))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

Shape of label tensor: (20229, 3)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 embedding (Embedding)       (None, 100, 128)          13552512  
                                                                 
 transformer_block (Transfo  (None, 100, 128)          561024    
 rmerBlock)                                                      
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)           